# CICCADA — Stage 2: Conformance Table Builders

Builds `conformance_voltvar_v2`, `conformance_voltwatt_v2`, `conformance_voltwattghi_v2`
from `ts` + `meta_up23c` + `all_uncurtailedpv_v2` (Stage 1 output).

**Run the cells in order.** Sections 1–4 are the smoke test on a single month /
single site-slice; do not skip to Section 5 until Section 4 comes back clean.

| Issue | Fixed in |
|---|---|
| R1 max(voltage) | `stage2_common.site_agg_cte` |
| R2 flex_export_detected = False | `stage2_common.meta_filter` (`exclude_flex=True`) |
| R3 / R9 AEST dates | `stage2_common.aest_month_window` + `temporal_cols` |
| R4 column naming | `build_conformance_voltvar` (thresholds unchanged) |
| R5 reduced non-conformance | emitted **both ways**, pending Baran |
| R7 V-VAr curtailment zone | `build_conformance_voltvar` |
| R10 capability on S_99 | `as4777_curves.q_cap_absorbing_sql('P_kW','S_99')` |
| R11 curve keystone | both builders import, none re-implement |
| R12 AEST day/night | `stage2_common.temporal_cols` |
| R13 total_count | `build_conformance_voltwatt` (both tables agree) |
| R14 null_uncurtailed_P_count | both GHI-joined tables |
| R15 NULL not 0 | all curtailment columns |
| R16 2024 + 2025 | Section 5 |


## 0. Setup

In [ ]:
import sys, os, time
from pathlib import Path

# --- point Python at shared/ and at this stage2 folder --------------------
ROOT = Path.cwd().parents[1]          # .../ciccada_analysis
sys.path.insert(0, str(ROOT / "shared"))
sys.path.insert(0, str(ROOT / "data_calc_write" / "stage2_conformance"))

from aws_config import aq, tables, databases
from ciccada_config import SA, SAI

import build_conformance_voltvar as vv
import build_conformance_voltwatt as vw
from stage2_common import aest_month_window

DB = SAI          # solar_analytics_iceberg
N_PARTS = 8       # site slices: site_id % N_PARTS

print("targets:", vv.TARGET, vw.TARGET_BASIC, vw.TARGET_GHI)

In [ ]:
# Connection + Stage 1 dependency check.
# Iceberg tables return nothing from DESCRIBE -- use SELECT * LIMIT 1.
aq("SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites "
   "FROM all_uncurtailedpv_v2", database=DB)

## 1. Read the SQL before you run it

`preview_sql` builds the exact INSERT for one slice without executing it.
Check the AEST window: for AEST January 2024 it should read UTC partitions
`(2023,12)` and `(2024,1)`, from `2023-12-31 14:00:00` to `2024-01-31 14:00:00`.
That two-partition read is the D2 fix — a UTC month is not an AEST month.

In [ ]:
print(aest_month_window(2024, 1))   # -> ('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023,12),(2024,1)])
print(aest_month_window(2024, 7))
print(aest_month_window(2025, 12))

In [ ]:
print(vv.preview_sql(year=2024, month=1, n_parts=N_PARTS, part=0)[:4000])

## 2. Create the empty tables

Destructive: drops and recreates. `_v2` suffix throughout — Hossein's originals
are never touched.

In [ ]:
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))
time.sleep(5)   # Glue catalog is eventually consistent
tables(DB)[tables(DB)["Table"].str.contains("conformance")][["Table"]]

## 3. Smoke test — one AEST month, one site slice

`parts=[0]` is 1/8 of sites for AEST January 2024. This should complete in a few
minutes. If it fails, it fails cheaply.

In [ ]:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=[0], exclude_flex=True)

In [ ]:
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=[0], exclude_flex=True)
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=[0], exclude_flex=True)

## 4. Validate the smoke test

Every "MUST be 0" line must actually be 0 before you go any further.

The one to watch is **duplicate keys**. If that is non-zero, the AEST window
logic has leaked and a site-day has been split across two INSERTs.

In [ ]:
vv.validate(aq, database=DB);

In [ ]:
vw.validate_basic(aq, database=DB);

In [ ]:
vw.validate_ghi(aq, database=DB);

In [ ]:
# Eyeball actual rows. Check: day spans 1..31, day_night has both values,
# and P_kW_sum is positive during the day.
aq(f"""
    SELECT site_id, year, month, day, day_night,
           round(P_kW_sum, 1)                   AS P_kW_sum,
           round(nonconformance_voltvar_sum, 3) AS nonconf,
           round(curtailment_voltvar_sum, 3)    AS curtail,
           curtailment_eligible_count, null_uncurtailed_P_count,
           exposed_count, all_intervals_count, total_count
    FROM {vv.TARGET}
    ORDER BY curtailment_voltvar_sum DESC NULLS LAST
    LIMIT 15
""", database=DB)

In [ ]:
# R3 regression test: the AEST boundary.
# Under Hossein's UTC extraction, intervals from 00:00-09:55 AEST were booked to
# the previous day. Here, day 1 of the month must contain a full AEST day --
# including its early-morning (night) intervals, which live in the PREVIOUS UTC
# month's partition. If day=1 has far fewer intervals than day=2, the window
# logic is wrong.
aq(f"""
    SELECT day, sum(all_intervals_count) AS intervals, count(DISTINCT site_id) AS sites
    FROM {vv.TARGET}
    WHERE year = 2024 AND month = 1 AND day IN (1, 2, 15, 30, 31)
    GROUP BY day ORDER BY day
""", database=DB)

## 5. Full load (R16: 2024 **and** 2025)

Each call is one AEST month × 8 site-slices = 8 Athena queries. A full year is
96 queries per table. Run one table at a time and check the printout.

If Athena throttles (`TooManyRequestsException`), drop `N_PARTS` to 4 or run
`months` in two halves.

In [ ]:
# Volt-VAr, 2024. Part 0 of Jan is already loaded from the smoke test --
# rerun it and you WILL double-count. Load Jan parts 1..7, then Feb-Dec fully.
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
vv.run_months_voltvar(aq, database=DB, year=2024, months=list(range(2, 13)),
                      n_parts=N_PARTS)

In [ ]:
vv.run_months_voltvar(aq, database=DB, year=2025, months=list(range(1, 13)),
                      n_parts=N_PARTS)

In [ ]:
# Volt-Watt basic -- same pattern (Jan part 0 already loaded)
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
vw.run_months_basic(aq, database=DB, year=2024, months=list(range(2, 13)), n_parts=N_PARTS)
vw.run_months_basic(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

In [ ]:
# Volt-Watt GHI -- R16: 2025 as well as 2024. The original only ever had 2024.
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
vw.run_months_ghi(aq, database=DB, year=2024, months=list(range(2, 13)), n_parts=N_PARTS)
vw.run_months_ghi(aq, database=DB, year=2025, months=list(range(1, 13)), n_parts=N_PARTS)

## 6. Full validation

In [ ]:
vv.validate(aq, database=DB);

In [ ]:
vw.validate_basic(aq, database=DB);
vw.validate_ghi(aq, database=DB);

In [ ]:
# R13 regression test: the two Volt-Watt tables must share a denominator.
vw.cross_check(aq, database=DB);

## 7. Reconcile against Hossein's tables

Differences are **expected**. They should be fully explained by:

- **R2** — flex-export sites excluded (biggest effect; ~700 sites in Stage 1)
- **R1** — `max(voltage)` is >= `avg(voltage)`, so more intervals cross 240 V / 253 V
- **R3** — AEST day boundaries reshuffle intervals between days
- **R10** — capability clamped on `s_99`, not nameplate

If the site-count drop does **not** match the flex-export count, stop and find
out why before trusting anything.

In [ ]:
vv.compare_to_original(aq, database=DB, original="conformance_voltvar");

In [ ]:
# Fleet-level headline comparison. Expect the same order of magnitude, not
# identical numbers.
aq(f"""
    SELECT 'v2' AS tbl, year,
           round(sum(nonconformance_voltvar_sum), 0)  AS nonconf_kvar,
           round(sum(curtailment_voltvar_sum), 0)     AS curtail_kw,
           sum(total_count)                           AS intervals
    FROM {vv.TARGET} GROUP BY year
    ORDER BY year
""", database=DB)

## 8. R5 — the number you cannot publish yet

`nonconformance_voltvar_red_sum` reproduces Hossein's definition:
`adverse + inactive + near_conformant` — i.e. it counts the sites that are
essentially complying and ignores the ones falling well short.

`nonconformance_voltvar_red_alt_sum` is `adverse + inactive + significant_shortfall`
— what the definition almost certainly should be.

Run this, take both numbers to Baran, and get him to confirm which range CANVAS
intended before either appears in the paper.

In [ ]:
aq(f"""
    SELECT year,
           round(sum(nonconformance_voltvar_red_sum), 0)     AS red_hossein,
           round(sum(nonconformance_voltvar_red_alt_sum), 0) AS red_alt,
           sum(nonconformance_voltvar_red_count)             AS red_hossein_count,
           sum(nonconformance_voltvar_red_alt_count)         AS red_alt_count,
           round(sum(Q_near_conformant_sum), 0)              AS near_conformant,
           round(sum(Q_significant_shortfall_sum), 0)        AS significant_shortfall
    FROM {vv.TARGET}
    GROUP BY year ORDER BY year
""", database=DB)